# Recommendation System

This notebook develops the content-based homestay recommendation system by:

1. Loading the prepared dataset.
2. Converting homestay feature representations into TF-IDF vectors.
3. Computing similarity scores using cosine similarity.
4. Generating personalized homestay recommendations.
5. Providing explainable recommendations based on shared features.
6. Testing the recommendation engine on sample homestays.

The generated recommendation model will be used for evaluation and deployment.

In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# ==========================================
# LOAD PREPARED DATASET
# ==========================================

df = pd.read_csv(
    "../data/final/homestays_prepared.csv"
)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (811, 45)


,homestay_id,homestay_name,owner_name,category,district,block,village,owner_email,owner_mobile,google_name,...,mountain_view,room_service,bonfire_barbeque,pickup_dropoff_service,description,price_band,amenity_text,location_text,proximity_text,feature_text
0,1,Revere Homestay,Mr. Riwaj Pradhan,Silver,Kalimpong,Municipality,"8Th Mile, Kalimpong",riwajpradhan10@gmail.com,9800686780,Revere Homsetay,...,1,0,0,0,"Revere Homestay, located in 8th Mile, Kalimpon...",mid_range,wifi parking breakfast mountain_view,"Municipality 8Th Mile, Kalimpong",deolo_moderately_close durpin_very_close town_...,"Revere Homestay, located in 8th Mile, Kalimpon..."
1,2,Mansarover Homestay,Miss Tina Mani Gurung,Gold,Kalimpong,Municipality,"Chandralok, Kalimpong",santabgurung53@gmail.com,9932234895,Mansarover Homestay / Flora & Transport,...,0,1,1,0,"Mansarover Homestay, located in Chandralok, Ka...",premium,wifi parking breakfast room_service bonfire_ba...,"Municipality Chandralok, Kalimpong",deolo_moderately_close durpin_very_close town_...,"Mansarover Homestay, located in Chandralok, Ka..."
2,3,Bethany Homestay,Anupama Tamang,Silver,Kalimpong,Kalimpong I,Dr.Grahams Home Block B,wangchuck20199@gmial.com,8348993048,Bethany Homestay Kalimpong,...,1,0,0,0,"BETHANY HOMESTAY, located in Dr.GRAHAMS HOME B...",budget,breakfast mountain_view,Kalimpong I Dr.Grahams Home Block B,deolo_very_close durpin_moderately_close town_...,"BETHANY HOMESTAY, located in Dr.GRAHAMS HOME B..."
3,5,Bajarangi Homestay,Kamal Kumar Sharma,Silver,Kalimpong,Kalimpong I,Singi Samalbong Kalimpong,bajrangihomestay@gmail.com,8670450557,Bajrangi Homestay,...,1,1,0,0,"BAJARANGI HOMESTAY, located in SINGI SAMALBONG...",mid_range,parking breakfast mountain_view room_service,Kalimpong I Singi Samalbong Kalimpong,deolo_moderately_close durpin_moderately_close...,"BAJARANGI HOMESTAY, located in SINGI SAMALBONG..."
4,7,Relly View Homestay,Soma Sundas,Silver,Kalimpong,Kalimpong I,Dr.Grahams Homes Block B Kalimpong,somasundas55@gmail.com,9932095235,Kalimpong View Home Stay,...,0,0,1,1,Relly View Homestay is located in DR.GRAHAMS H...,mid_range,wifi breakfast bonfire_barbeque pickup_dropoff...,Kalimpong I Dr.Grahams Homes Block B Kalimpong,deolo_very_close durpin_moderately_close town_...,Relly View Homestay is located in DR.GRAHAMS H...


In [3]:
# ==========================================
# FEATURE TEXT VALIDATION
# ==========================================

print(
    df.loc[0, "feature_text"]
)

Revere Homestay, located in 8th Mile, Kalimpong, block Municipality, is situated very close to town, moderately close to Deolo, and very close to Durpin. This Silver-rated homestay offers Wifi, Parking, Breakfast, and Mountain View. Rated 5.0 with 1 review, it provides a convenient stay with proximity to all three locations. wifi parking breakfast mountain_view Municipality 8Th Mile, Kalimpong deolo_moderately_close durpin_very_close town_very_close lava_far pedong_far gorubathan_far rishop_far lolegaon_far Silver mid_range


In [4]:
# ==========================================
# TF-IDF VECTORIZATION
# ==========================================
# Two fixes here, both evidenced by real output from a previous run:
# - "Wi-Fi" was splitting into the tokens "wi" and "fi" (hyphen breaks
#   the default tokenizer); "drop-off"/"pick-up" had the same problem.
#   A preprocessor normalizes these before tokenization.
# - Generic LLM-boilerplate words ("including", "key", "locations",
#   "offers", etc.) were showing up as top matching terms despite
#   carrying no real distinguishing information -- extended stop words
#   to filter them out.

import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"wi[\s-]*fi", "wifi", text)
    text = re.sub(r"drop[\s-]*off", "dropoff", text)
    text = re.sub(r"pick[\s-]*up", "pickup", text)
    return text

DOMAIN_STOP_WORDS = [
    "including", "key", "locations", "location", "offers", "offering",
    "provides", "providing", "features", "featuring", "enjoy", "ensuring",
    "ensures", "convenient", "reliable", "situated", "nestled", "guests",
    "stay", "homestay",
    # "kalimpong" is baked into many village names and into most
    # LLM-generated descriptions directly (not just the district field,
    # which was already removed in notebook 04) -- since this is a
    # single-district case study, the term is close to universal and
    # carries no real distinguishing signal regardless of which field
    # it comes from.
    "kalimpong",
    # Proximity/rating boilerplate: nearly every description mentions
    # "close" (from Very Close/Moderately Close labels -- most homestays
    # are close to at least one of 8 tourist locations) and "rated"
    # (from "Rated X.X with Y reviews" phrasing). Neither says WHICH
    # location or WHAT rating -- that specific information is carried
    # by the actual place names and numbers, which stay in vocabulary.
    "close", "far", "moderately", "rated",
]

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
custom_stop_words = list(ENGLISH_STOP_WORDS) + DOMAIN_STOP_WORDS

tfidf = TfidfVectorizer(
    stop_words=custom_stop_words,
    preprocessor=normalize_text,
)

tfidf_matrix = tfidf.fit_transform(
    df["feature_text"]
)

print(
    "TF-IDF Matrix Shape:",
    tfidf_matrix.shape
)

TF-IDF Matrix Shape: (811, 1766)


In [5]:
# ==========================================
# VOCABULARY DOCUMENT-FREQUENCY CHECK
# ==========================================
# Surfaces any remaining near-universal terms (appearing in most
# documents) that survived the stop-word list -- these carry weak
# discriminative signal for TF-IDF regardless of what they mean. Rather
# than wait for one to show up in a sample explanation, check directly.

import numpy as np

doc_freq = np.asarray((tfidf_matrix > 0).sum(axis=0)).flatten()
n_docs = tfidf_matrix.shape[0]

df_check = pd.DataFrame({
    "term": tfidf.get_feature_names_out(),
    "doc_frequency": doc_freq,
    "pct_of_documents": (doc_freq / n_docs * 100).round(1),
}).sort_values("doc_frequency", ascending=False)

print("Top 20 most common terms remaining in vocabulary:")
print(df_check.head(20).to_string(index=False))

high_freq = df_check[df_check["pct_of_documents"] > 40]
if len(high_freq) > 0:
    print(f"\n{len(high_freq)} terms appear in over 40% of documents -- worth reviewing whether they carry real signal:")
    print(high_freq.to_string(index=False))
else:
    print("\nNo terms appear in over 40% of documents.")

Top 20 most common terms remaining in vocabulary:
          term  doc_frequency  pct_of_documents
         deolo            808              99.6
        durpin            807              99.5
          town            804              99.1
gorubathan_far            789              97.3
        silver            789              97.3
       reviews            782              96.4
       located            699              86.2
        budget            644              79.4
     breakfast            643              79.3
       parking            618              76.2
         block            612              75.5
  lolegaon_far            592              73.0
          wifi            581              71.6
    durpin_far            568              70.0
      town_far            509              62.8
 mountain_view            469              57.8
    rishop_far            466              57.5
    pedong_far            453              55.9
      lava_far            443         

In [6]:
# ==========================================
# COSINE SIMILARITY MATRIX
# ==========================================

cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

print(
    "Similarity Matrix Shape:",
    cosine_sim.shape
)

Similarity Matrix Shape: (811, 811)


In [7]:
# ==========================================
# HOMESTAY INDEX MAPPING
# ==========================================
# Previously indexed by homestay_name with .drop_duplicates() -- this
# silently discarded every homestay sharing a name with an earlier one,
# keeping only one. Confirmed real duplicate names exist in this dataset
# (e.g. "Green Valley Homestay" x5, genuinely different properties), so
# that bug made most of them permanently unreachable. homestay_id is
# guaranteed unique (verified in notebook 04) and is now the lookup key.

assert df["homestay_id"].is_unique, "homestay_id must be unique for lookups to be unambiguous."

indices = pd.Series(
    df.index.values,
    index=df["homestay_id"]
)

print(
    "Total Homestays:",
    len(indices)
)


def find_homestay_ids_by_name(name):
    """
    Name-based search for convenience -- returns ALL matches (not just
    one), since names aren't unique. Use the returned homestay_id with
    recommend_homestays() / explainable_recommendations() to disambiguate.
    """
    matches = df[df["homestay_name"].str.lower() == name.lower()]
    return matches[["homestay_id", "homestay_name", "village", "block"]]

Total Homestays: 811


In [8]:
# ==========================================
# RECOMMENDATION FUNCTION
# ==========================================

def recommend_homestays(
    homestay_id,
    top_n=5
):

    idx = indices[homestay_id]

    similarity_scores = list(
        enumerate(
            cosine_sim[idx]
        )
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[
        1:top_n + 1
    ]

    homestay_indices = [
        i[0]
        for i in similarity_scores
    ]

    scores = [
        round(i[1], 3)
        for i in similarity_scores
    ]

    recommendations = (
        df.iloc[homestay_indices]
        .copy()
    )

    recommendations[
        "similarity_score"
    ] = scores

    return recommendations

In [9]:
# ==========================================
# TF-IDF EXPLAINABILITY FUNCTIONS
# ==========================================

feature_names = tfidf.get_feature_names_out()

def get_top_matching_terms(
    source_idx,
    recommended_idx,
    top_n=5
):

    source_vector = (
        tfidf_matrix[source_idx]
        .toarray()[0]
    )

    recommended_vector = (
        tfidf_matrix[recommended_idx]
        .toarray()[0]
    )

    contributions = (
        source_vector
        *
        recommended_vector
    )

    top_indices = (
        contributions.argsort()
        [-top_n:]
        [::-1]
    )

    terms = [
        feature_names[i]
        for i in top_indices
        if contributions[i] > 0
    ]

    return terms


def generate_explanation(
    source_row,
    recommended_row,
    source_idx,
    recommended_idx
):

    reasons = []

    # ==========================
    # Common Amenities
    # ==========================

    amenity_columns = [

        "wifi",
        "parking",
        "breakfast",
        "mountain_view",
        "room_service",
        "bonfire_barbeque",
        "pickup_dropoff_service"

    ]

    common_features = []

    for amenity in amenity_columns:

        if (
            source_row[amenity] == 1
            and
            recommended_row[amenity] == 1
        ):

            common_features.append(
                amenity.replace("_", " ")
            )

    # ==========================
    # Location Similarity
    # ==========================

    if (
        source_row["block"]
        ==
        recommended_row["block"]
    ):
        reasons.append(
            f"Both are located in {source_row['block']}"
        )

    # ==========================
    # Category Similarity
    # ==========================

    if (
        source_row["category"]
        ==
        recommended_row["category"]
    ):
        reasons.append(
            f"Same category ({source_row['category']})"
        )

    # ==========================
    # Rating Similarity
    # ==========================
    # Skipped if either homestay's location was manually corrected in
    # 02a -- rating/review_count for those rows still belong to the
    # original wrong Google match and aren't verified.

    rating_is_reliable = (
        not source_row.get("location_corrected", False)
        and not recommended_row.get("location_corrected", False)
    )

    if (
        rating_is_reliable
        and abs(
            source_row["rating"]
            -
            recommended_row["rating"]
        ) <= 0.5
    ):

        reasons.append(
            "Similar ratings"
        )

    # ==========================
    # Proximity Similarity
    # ==========================
    # Exact proximity-token matches (e.g. both "durpin_far") DO count
    # toward the TF-IDF similarity score, but very specific address
    # fragments often get weighted higher (higher IDF, rarer terms) and
    # can crowd proximity matches out of the top-5 displayed terms even
    # when the match is real and substantial. This makes a genuine
    # majority-match explicit and human-readable regardless of whether
    # it wins the raw term ranking.

    proximity_columns = [
        "deolo_proximity", "durpin_proximity", "town_proximity",
        "lava_proximity", "pedong_proximity", "gorubathan_proximity",
        "rishop_proximity", "lolegaon_proximity",
    ]

    matching_proximity = [
        col.replace("_proximity", "").title()
        for col in proximity_columns
        if source_row[col] == recommended_row[col]
    ]

    # Threshold set at more than half (5 of 8) -- with 3 possible
    # categories per location, a couple of matches could be coincidental,
    # but a majority match reflects genuine physical closeness between
    # the two homestays themselves.
    if len(matching_proximity) >= 5:
        reasons.append(
            f"Similar proximity to {', '.join(matching_proximity)} "
            f"({len(matching_proximity)}/8 locations)"
        )

    # ==========================
    # Price Similarity
    # ==========================
    # Not gated by location_corrected -- price is synthetically
    # generated (category-based), not Google-derived, so it's
    # unaffected by the location-match issue.

    if abs(
        source_row["price"]
        -
        recommended_row["price"]
    ) <= 1000:

        reasons.append(
            "Similar price range"
        )

    # ==========================
    # TF-IDF Terms
    # ==========================

    tfidf_terms = get_top_matching_terms(
        source_idx,
        recommended_idx
    )

    return (
        common_features,
        reasons,
        tfidf_terms
    )

In [10]:
# ==========================================
# EXPLAINABLE RECOMMENDATIONS
# ==========================================

def explainable_recommendations(
    homestay_id,
    top_n=5
):

    recommendations = recommend_homestays(
        homestay_id,
        top_n
    )

    source_idx = indices[
        homestay_id
    ]

    source_row = df.iloc[
        source_idx
    ]

    print(
        f"\nSelected Homestay: {source_row['homestay_name']} (ID: {homestay_id})"
    )

    print("=" * 80)

    for _, row in recommendations.iterrows():

        recommended_idx = row.name

        print(
            f"\nRecommended: {row['homestay_name']} (ID: {row['homestay_id']})"
        )

        print(
            f"Similarity Score: "
            f"{row['similarity_score']:.3f}"
        )

        print(
            f"Rating: {row['rating']}"
        )

        print(
            f"Price: \u20b9{row['price']}"
        )

        (
            common_features,
            additional_reasons,
            tfidf_terms
        ) = generate_explanation(
            source_row,
            row,
            source_idx,
            recommended_idx
        )

        print("\nCommon Features:")

        if common_features:

            for feature in common_features:

                print(
                    f"\u2713 {feature.title()}"
                )

        print("\nModel Explanation:")

        if tfidf_terms:

            print(
                "Top matching TF-IDF terms:"
            )

            for term in tfidf_terms:

                print(
                    f"\u2022 {term}"
                )

        print("\nAdditional Similarities:")

        for reason in additional_reasons:

            print(
                f"\u2713 {reason}"
            )

        # =====================
        # Natural Language XAI
        # =====================

        feature_text = ", ".join(
            common_features[:3]
        )

        term_text = ", ".join(
            tfidf_terms[:5]
        )

        explanation = (

            f"{row['homestay_name']} was "
            f"recommended because it shares "
            f"similar characteristics with "
            f"{source_row['homestay_name']}, including "
            f"{feature_text}. "

            f"The recommendation is further "
            f"supported by similar descriptive "
            f"terms such as {term_text}."

        )

        print(
            "\nNatural Language Explanation:"
        )

        print(explanation)

        print("-" * 80)

In [11]:
# ==========================================
# TEST RECOMMENDATIONS
# ==========================================

sample_homestay_id = (
    df["homestay_id"]
    .iloc[0]
)

recommend_homestays(
    sample_homestay_id
)[
    [
        "homestay_id",
        "homestay_name",
        "rating",
        "price",
        "similarity_score"
    ]
]

,homestay_id,homestay_name,rating,price,similarity_score
115,156,The Birds View Homestay,4.6,4466,0.573
126,167,Windsong Homestay,4.5,3538,0.512
318,423,The Scarlett,5.0,1920,0.308
736,1062,Kanchanjunga Homestay,4.3,1803,0.308
125,166,Pranati Residency Homestay,4.9,1206,0.296


In [18]:
# ==========================================
# TEST EXPLAINABILITY
# ==========================================

sample_homestay_id = (
    df["homestay_id"]
    .iloc[810]
)

explainable_recommendations(
    sample_homestay_id
)


Selected Homestay: Eco Hill Homestay (ID: 1157)

Recommended: Mountain View Homestay (ID: 808)
Similarity Score: 0.579
Rating: 4.8
Price: ₹1089

Common Features:
✓ Parking
✓ Breakfast

Model Explanation:
Top matching TF-IDF terms:
• tendrabong
• busty
• pedong
• parking
• breakfast

Additional Similarities:
✓ Both are located in Pedong
✓ Same category (Silver)
✓ Similar ratings
✓ Similar price range

Natural Language Explanation:
Mountain View Homestay was recommended because it shares similar characteristics with Eco Hill Homestay, including parking, breakfast. The recommendation is further supported by similar descriptive terms such as tendrabong, busty, pedong, parking, breakfast.
--------------------------------------------------------------------------------

Recommended: Golay Homestay (ID: 394)
Similarity Score: 0.548
Rating: 5.0
Price: ₹2139

Common Features:
✓ Breakfast
✓ Bonfire Barbeque

Model Explanation:
Top matching TF-IDF terms:
• tendrabong
• busty
• pedong
• experienc

In [13]:
# ==========================================
# PREFERENCE FILTERING
# ==========================================

def search_by_preferences(

    mountain_view=False,
    breakfast=False,
    parking=False,
    block=None

):

    results = df.copy()

    if mountain_view:

        results = results[
            results["mountain_view"] == 1
        ]

    if breakfast:

        results = results[
            results["breakfast"] == 1
        ]

    if parking:

        results = results[
            results["parking"] == 1
        ]

    if block:

        results = results[
            results["block"]
            ==
            block
        ]

    return results[
        [
            "homestay_id",
            "homestay_name",
            "rating",
            "price",
            "block"
        ]
    ].sort_values(
        by="rating",
        ascending=False
    )

In [14]:
search_by_preferences(
    mountain_view=True,
    breakfast=True,
    parking=True
).head()

,homestay_id,homestay_name,rating,price,block
810,1157,Eco Hill Homestay,5.0,1503,Pedong
0,1,Revere Homestay,5.0,2126,Municipality
6,9,Bhattarai Homestay,5.0,1121,Kalimpong I
793,1136,Tenzing Tranquil Farmstay,5.0,1822,Kalimpong I
792,1135,Alstroemeria Homestay,5.0,1349,Kalimpong I


In [15]:
# ==========================================
# SAVE MODEL FILES
# ==========================================

with open(
    "../models/tfidf_vectorizer.pkl",
    "wb"
) as f:

    pickle.dump(
        tfidf,
        f
    )

with open(
    "../models/cosine_similarity.pkl",
    "wb"
) as f:

    pickle.dump(
        cosine_sim,
        f
    )

with open(
    "../models/indices.pkl",
    "wb"
) as f:

    pickle.dump(
        indices,
        f
    )

with open(
    "../models/tfidf_matrix.pkl",
    "wb"
) as f:

    pickle.dump(
        tfidf_matrix,
        f
    )
    
print(
    "Model files saved successfully."
)

Model files saved successfully.
